# Preprocessing MIMICIV Data without using PyHealth

In [40]:
import pandas as pd

# ------------------------- STEP 1: LOAD DATA -------------------------

# Load patient demographics
patients = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/patients.csv")

# Load hospital admissions
admissions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/admissions.csv")

# Load ICU stays
icustays = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/icu/icustays.csv")

# Load diagnoses (ICD codes assigned to patients)
diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/diagnoses_icd.csv")

# Load prescriptions (medications administered)
prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")

# Load medical procedures (ICD codes for treatments)
procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/procedures_icd.csv")

# Load ICD descriptions (for both ICD-9 and ICD-10)
icd_diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_diagnoses.csv.gz")
icd_procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_procedures.csv.gz")

# Convert ICU stay timestamps to datetime format
icustays['intime'] = pd.to_datetime(icustays['intime'], errors='coerce')
icustays['outtime'] = pd.to_datetime(icustays['outtime'], errors='coerce')



/tmp/ipykernel_4291/2524248172.py:18: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")


In [41]:
diagnoses.head(5)

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9


In [42]:
icd_diagnoses.head(5)

,icd_code,icd_version,long_title
0,0010,9,Cholera due to vibrio cholerae
1,0011,9,Cholera due to vibrio cholerae el tor
2,0019,9,"Cholera, unspecified"
3,0020,9,Typhoid fever
4,0021,9,Paratyphoid fever A


In [17]:
procedures.head(5)

,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version_x,icd_version_y,long_title,procedure_description
0,10000032,22595853,1,2180-05-07,5491,9,9,Percutaneous abdominal drainage,Percutaneous abdominal drainage
1,10000032,22841357,1,2180-06-27,5491,9,9,Percutaneous abdominal drainage,Percutaneous abdominal drainage
2,10000032,25742920,1,2180-08-06,5491,9,9,Percutaneous abdominal drainage,Percutaneous abdominal drainage
3,10000068,25022803,1,2160-03-03,8938,9,9,Other nonoperative respiratory measurements,Other nonoperative respiratory measurements
4,10000117,27988844,1,2183-09-19,0QS734Z,10,10,Reposition Left Upper Femur with Internal Fixa...,Reposition Left Upper Femur with Internal Fixa...


In [48]:
# ------------------------- STEP 2: MAP ICD CODES TO DESCRIPTIONS -------------------------

# Merge diagnoses with ICD descriptions
diagnoses = diagnoses.merge(icd_diagnoses, on="icd_code", how="left").rename(columns={'icd_version_x': 'icd_version'})
diagnoses['diagnosis_description'] = diagnoses['long_title'].fillna("Unknown diagnosis")

# Merge procedures with ICD descriptions
procedures = procedures.merge(icd_procedures, on="icd_code", how="left").rename(columns={'icd_version_x': 'icd_version'})
procedures['procedure_description'] = procedures['long_title'].fillna("Unknown procedure")

In [50]:
# Keep only necessary columns
diagnoses = diagnoses[['subject_id', 'hadm_id', 'icd_version', 'diagnosis_description']]
procedures = procedures[['subject_id', 'hadm_id', 'icd_version', 'procedure_description']]

In [51]:
patients.head(5)

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


In [52]:
# ------------------------- STEP 3: COMPUTE AGE AT EVENTS -------------------------

# Keep only relevant patient information
patients = patients[['subject_id', 'anchor_age', 'anchor_year']]

# Merge patient age into all event tables
admissions = admissions.merge(patients, on='subject_id', how='left', suffixes=None)
icustays = icustays.merge(patients, on='subject_id', how='left', suffixes=None)
diagnoses = diagnoses.merge(patients, on='subject_id', how='left', suffixes=None)
procedures = procedures.merge(patients, on='subject_id', how='left', suffixes=None)
prescriptions = prescriptions.merge(patients, on='subject_id', how='left', suffixes=None)

# Convert date columns
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
icustays['intime'] = pd.to_datetime(icustays['intime'])
prescriptions['starttime'] = pd.to_datetime(prescriptions['starttime'])

# Function to compute patient age at event time
def compute_age_at_event(df, event_col):
    df['event_year'] = df[event_col].dt.year
    df['age_at_event'] = df['anchor_age'] + (df['event_year'] - df['anchor_year'])
    return df

# Apply age computation for events with timestamps
admissions = compute_age_at_event(admissions, 'admittime')
icustays = compute_age_at_event(icustays, 'intime')
prescriptions = compute_age_at_event(prescriptions, 'starttime')

# Directly assign age for diagnoses and procedures (since they don't have exact dates)
diagnoses['age_at_event'] = diagnoses['anchor_age']
procedures['age_at_event'] = procedures['anchor_age']

/tmp/ipykernel_4291/535772561.py:7: FutureWarning: Passing 'suffixes' as a <class 'NoneType'>, is not supported and may give unexpected results. Provide 'suffixes' as a tuple instead. In the future a 'TypeError' will be raised.
  admissions = admissions.merge(patients, on='subject_id', how='left', suffixes=None)
/tmp/ipykernel_4291/535772561.py:8: FutureWarning: Passing 'suffixes' as a <class 'NoneType'>, is not supported and may give unexpected results. Provide 'suffixes' as a tuple instead. In the future a 'TypeError' will be raised.
  icustays = icustays.merge(patients, on='subject_id', how='left', suffixes=None)
/tmp/ipykernel_4291/535772561.py:9: FutureWarning: Passing 'suffixes' as a <class 'NoneType'>, is not supported and may give unexpected results. Provide 'suffixes' as a tuple instead. In the future a 'TypeError' will be raised.
  diagnoses = diagnoses.merge(patients, on='subject_id', how='left', suffixes=None)
/tmp/ipykernel_4291/535772561.py:10: FutureWarning: Passing 'suf

In [54]:
# ------------------------- STEP 4: GENERATE STRUCTURED CLINICAL HISTORY -------------------------

def generate_patient_history(subject_id):
    """Generates a structured clinical history for a given patient."""

    history = [f"Patient {subject_id} Clinical History:\n"]

    # Get all hospitalizations for this patient
    patient_admissions = admissions[admissions['subject_id'] == subject_id].sort_values('admittime')

    # Group hospitalizations by age
    for age, adm_group in patient_admissions.groupby('age_at_event'):

        history.append(f"\n--- Age {int(age)} ---")

        for _, adm in adm_group.iterrows():
            event_type = "Emergency" if adm['admission_type'] in ["EMERGENCY", "URGENT"] else "Normal"
            history.append(f"\nHospitalization ({event_type}) - Discharge status: {adm['discharge_location']}")

            # Diagnoses for this hospitalization
            patient_diagnoses = diagnoses[(diagnoses['subject_id'] == subject_id) & (diagnoses['hadm_id'] == adm['hadm_id'])]
            if not patient_diagnoses.empty:
                history.append("\n  Diagnoses:")
                for _, diag in patient_diagnoses.iterrows():
                    history.append(f"    - {diag['diagnosis_description']}")

            # Procedures performed
            patient_procedures = procedures[(procedures['subject_id'] == subject_id) & (procedures['hadm_id'] == adm['hadm_id'])]
            if not patient_procedures.empty:
                history.append("\n  Procedures:")
                for _, proc in patient_procedures.iterrows():
                    history.append(f"    - {proc['procedure_description']}")

            # Medications prescribed
            patient_meds = prescriptions[(prescriptions['subject_id'] == subject_id) & (prescriptions['hadm_id'] == adm['hadm_id'])]
            if not patient_meds.empty:
                history.append("\n  Medications Prescribed:")
                for _, med in patient_meds.iterrows():
                    history.append(f"    - {med['drug']}")

            # ICU stay duration calculation with safe handling for missing outtime
            patient_icu = icustays[(icustays['subject_id'] == subject_id) & (icustays['hadm_id'] == adm['hadm_id'])]
            for _, icu in patient_icu.iterrows():
                if pd.notna(icu['outtime']):  # Ensure 'outtime' is not missing
                    icu_stay_length = round((icu['outtime'] - icu['intime']).days, 1)
                    history.append(f"    - ICU admission for {icu_stay_length} days")
                else:
                    history.append(f"    - ICU admission (discharge date unknown)")

    return "\n".join(history)


# ------------------------- STEP 5: GENERATE AND SAVE SUMMARIES -------------------------

# Process multiple patients (limit to 100 patients for testing)
patient_ids = patients['subject_id'].unique()
summaries = {pid: generate_patient_history(pid) for pid in patient_ids[:100]}

# Save all summaries to a file
with open("clinical_histories.txt", "w") as file:
    for summary in summaries.values():
        file.write(summary + "\n\n")


In [63]:
admissions

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag,anchor_age,anchor_year,event_year,age_at_event
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0,52,2180,2180,52
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0,52,2180,2180,52
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0,52,2180,2180,52
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0,52,2180,2180,52
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0,19,2160,2160,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
546023,19999828,25744818,2149-01-08 16:44:00,2149-01-18 17:00:00,NaN,EW EMER.,P13JMH,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,English,SINGLE,WHITE,2149-01-08 09:11:00,2149-01-08 18:12:00,0,46,2147,2149,48
546024,19999828,29734428,2147-07-18 16:23:00,2147-08-04 18:10:00,NaN,EW EMER.,P38XL8,PHYSICIAN REFERRAL,HOME HEALTH CARE,Medicaid,English,SINGLE,WHITE,2147-07-17 17:18:00,2147-07-18 17:34:00,0,46,2147,2147,46
546025,19999840,21033226,2164-09-10 13:47:00,2164-09-17 13:42:00,2164-09-17 13:42:00,EW EMER.,P33612,EMERGENCY ROOM,DIED,Private,English,WIDOWED,WHITE,2164-09-10 11:09:00,2164-09-10 14:46:00,1,58,2164,2164,58
546026,19999840,26071774,2164-07-25 00:27:00,2164-07-28 12:15:00,NaN,EW EMER.,P036NA,EMERGENCY ROOM,HOME,Private,English,WIDOWED,WHITE,2164-07-24 21:16:00,2164-07-25 01:20:00,0,58,2164,2164,58


In [62]:
patient_ids[:100]

array([10000032, 10000048, 10000058, 10000068, 10000084, 10000102,
       10000108, 10000115, 10000117, 10000161, 10000178, 10000248,
       10000280, 10000285, 10000459, 10000473, 10000492, 10000507,
       10000526, 10000560, 10000594, 10000635, 10000650, 10000683,
       10000690, 10000719, 10000764, 10000826, 10000883, 10000886,
       10000891, 10000898, 10000904, 10000935, 10000947, 10000951,
       10000955, 10000969, 10000980, 10000995, 10001016, 10001038,
       10001122, 10001176, 10001186, 10001217, 10001319, 10001336,
       10001338, 10001401, 10001472, 10001492, 10001523, 10001565,
       10001574, 10001624, 10001629, 10001658, 10001663, 10001667,
       10001725, 10001757, 10001765, 10001823, 10001843, 10001851,
       10001860, 10001877, 10001884, 10001919, 10002011, 10002012,
       10002013, 10002113, 10002114, 10002131, 10002147, 10002155,
       10002157, 10002167, 10002177, 10002191, 10002221, 10002266,
       10002315, 10002348, 10002384, 10002425, 10002428, 10002

In [73]:
# Display one sample summary
print(list(summaries.values())[21])

Patient 10000635 Clinical History:


--- Age 74 ---

Hospitalization (Normal) - Discharge status: nan

  Diagnoses:
    - Other specified cardiac dysrhythmias
    - Diabetes mellitus without mention of complication, type II or unspecified type, not stated as uncontrolled
    - Unspecified essential hypertension

  Procedures:
    - Excision or destruction of other lesion or tissue of heart, endovascular approach
    - Intracardiac echocardiography
    - Cardiac mapping

  Medications Prescribed:
    - Potassium Chloride
    - PNEUMOcoccal 23-valent polysaccharide vaccine
    - Dextrose 50%
    - Glucagon
    - Triamterene-HCTZ (37.5/25)
    - Acetaminophen
    - Simethicone
    - Omeprazole
    - Potassium Chloride
    - Aspirin EC
    - Sodium Chloride 0.9%  Flush
    - Influenza Virus Vaccine
    - Potassium Chloride
    - Glucose Gel
    - MetFORMIN (Glucophage)

--- Age 81 ---

Hospitalization (Normal) - Discharge status: nan

  Diagnoses:
    - Other chest pain
    - Facial weakne